# Tiny instruction-interface demo (image + task -> manipulable manual)

Give **an image + a task**, get a **step-by-step manual rendered on your real photo**, inside a
digital interface you can **manipulate** - including pressing two things together to join them.

What it demonstrates:
- **Decompose**: task + image -> short ordered atomic steps.
- **Ground (faithful)**: each step drawn as a box/dot on the *real* photo (no re-render, no drift).
- **Flip**: next/prev through steps.
- **Tap-to-deepen**: click an object -> it crops and explains that region.
- **Connect (press to join)**: press the source object, then press the destination -> the interface
  moves those pixels together, marks them "joined", and erases the source's old spot.
- **Multi-image context**: upload a close-up (e.g. a label) -> the model reads it and adds a note.
- **Correct**: switch to "Set highlight" and click to move a highlight by hand.

Model: **Moondream2** (~1.8B). Runs on a free Kaggle **T4 GPU**.
**Before running: Settings -> Accelerator -> GPU T4, and Internet -> On.**

In [ ]:
# 1. Install deps (Kaggle already has torch/torchvision)
!pip install -q -U transformers accelerate einops gradio pillow

In [ ]:
# 2. Load the small model (Moondream2)
import torch, re
from transformers import AutoModelForCausalLM
from PIL import Image, ImageDraw, ImageFilter
import gradio as gr

# If load ever breaks on the latest code, pin a revision, e.g. revision="2025-04-14"
model = AutoModelForCausalLM.from_pretrained(
    "vikhyatk/moondream2",
    trust_remote_code=True,
    device_map={"": "cuda"},
)
print("model loaded on", next(model.parameters()).device)

In [ ]:
# 3. Core logic: decompose, ground, draw, and connect

def generate_steps(image, task, max_steps=5):
    prompt = ('I want to: "' + task + '". Looking at this image, list the physical steps to do it '
              'as a short numbered list. One short single-action sentence per step. Max 5 steps.')
    ans = model.query(image, prompt)["answer"]
    steps = []
    for line in ans.split("\n"):
        line = re.sub(r"^[\-\*\d\.\)\s]+", "", line.strip()).strip()
        if line:
            steps.append(line)
    return steps[:max_steps] or [ans.strip()]

def step_object(image, step):
    q = ('For this instruction step, name the single main object to interact with, '
         'in 1 to 3 words only: "' + step + '"')
    return model.query(image, q)["answer"].strip().strip(".")

def ground(image, obj):
    try:
        objs = model.detect(image, obj).get("objects", [])
        if objs:
            return "box", objs
    except Exception:
        pass
    try:
        pts = model.point(image, obj).get("points", [])
        if pts:
            return "point", pts
    except Exception:
        pass
    return "none", []

def draw_overlay(base, kind, data, label):
    img = base.convert("RGB").copy()
    d = ImageDraw.Draw(img)
    W, H = img.size
    w = max(3, W // 200)
    if kind == "box":
        for b in data:
            d.rectangle([b["x_min"]*W, b["y_min"]*H, b["x_max"]*W, b["y_max"]*H],
                        outline=(255, 60, 60), width=w)
    elif kind == "point":
        for p in data:
            x, y = p["x"]*W, p["y"]*H
            r = max(8, W // 40)
            d.ellipse([x-r, y-r, x+r, y+r], outline=(255, 60, 60), width=w)
    d.rectangle([0, 0, W, 26], fill=(0, 0, 0))
    d.text((6, 7), label[:90], fill=(255, 255, 255))
    return img

def render(canvas, steps, idx):
    if not steps:
        return canvas, "No steps yet."
    step = steps[idx]
    obj = step_object(canvas, step)
    kind, data = ground(canvas, obj)
    view = draw_overlay(canvas, kind, data, "Step " + str(idx+1) + "/" + str(len(steps)) + ": " + obj)
    return view, "Step " + str(idx+1) + "/" + str(len(steps)) + ":  " + step

def erase_region(img, cx, cy, r):
    W, H = img.size
    box = (max(0, cx-r), max(0, cy-r), min(W, cx+r), min(H, cy+r))
    blurred = img.filter(ImageFilter.GaussianBlur(14)).crop(box)
    img.paste(blurred, box)

def connect(canvas, sx, sy, dx, dy):
    # move the source patch onto the destination, erase the old spot, mark as joined
    img = canvas.convert("RGB").copy()
    W, H = img.size
    r = min(W, H) // 7
    src_box = (max(0, sx-r), max(0, sy-r), min(W, sx+r), min(H, sy+r))
    patch = img.crop(src_box)
    erase_region(img, sx, sy, r)                      # old spot disappears
    pw, ph = patch.size
    dx0, dy0 = max(0, dx-pw//2), max(0, dy-ph//2)
    img.paste(patch, (dx0, dy0))                      # snapped onto destination
    d = ImageDraw.Draw(img)
    d.rectangle([dx0, dy0, dx0+pw, dy0+ph], outline=(60, 220, 120), width=max(3, W//200))
    d.text((dx0, max(0, dy0-14)), "joined", fill=(60, 220, 120))
    return img

print("logic ready")

In [ ]:
# 4. The manipulable interface (Gradio)

def on_generate(image, task):
    if image is None or not task or not task.strip():
        return None, "Upload an image and type a task first.", [], 0, None, None
    base = image.convert("RGB")
    steps = generate_steps(base, task)
    view, label = render(base, steps, 0)
    # outputs: step_view, label, steps_state, idx_state, base_state, canvas_state
    return view, label, steps, 0, base, base

def nav(delta, steps, idx, canvas):
    if not steps or canvas is None:
        return gr.update(), gr.update(), idx
    idx = max(0, min(len(steps)-1, idx + delta))
    view, label = render(canvas, steps, idx)
    return view, label, idx

def reset_canvas(base):
    if base is None:
        return gr.update(), None, None
    return base, base, None  # step_view, canvas_state, pending_state

def on_click(canvas, mode, pending, evt: gr.SelectData):
    # returns: step_view, deep_img, deep_txt, canvas_state, pending_state
    if canvas is None:
        return gr.update(), gr.update(), "", canvas, None
    x, y = evt.index
    W, H = canvas.size

    if mode == "Connect (press to join)":
        if pending is None:
            img = canvas.convert("RGB").copy()
            d = ImageDraw.Draw(img)
            r = max(6, min(W, H)//45)
            d.ellipse([x-r, y-r, x+r, y+r], fill=(60, 220, 120))
            d.text((x+r+2, y), "now press the destination", fill=(60, 220, 120))
            return img, gr.update(), "Source picked. Press where it should join.", canvas, (x, y)
        sx, sy = pending
        joined = connect(canvas, sx, sy, x, y)
        return joined, gr.update(), "Joined - two presses made one connection.", joined, None

    if mode == "Set highlight (correct)":
        img = canvas.convert("RGB").copy()
        d = ImageDraw.Draw(img)
        r = min(W, H) // 8
        d.rectangle([x-r, y-r, x+r, y+r], outline=(60, 160, 255), width=max(3, W//200))
        d.text((max(0, x-r), max(0, y-r-14)), "you set this", fill=(60, 160, 255))
        return img, gr.update(), "Highlight moved by hand (the human grounding signal).", canvas, None

    # Ask about object / deepen
    r = min(W, H) // 6
    crop = canvas.crop((max(0, x-r), max(0, y-r), min(W, x+r), min(H, y+r)))
    cap = model.query(crop, "What is this and what should I do with it here? One short sentence.")["answer"]
    return gr.update(), crop, cap, canvas, None

def on_context(cimg):
    if cimg is None:
        return ""
    note = model.query(cimg.convert("RGB"),
                       "Read any visible text or labels and give the single most useful detail "
                       "for following an instruction. One sentence.")["answer"]
    return "Context added: " + note

with gr.Blocks(title="Instruction interface demo") as demo:
    gr.Markdown("## Image + task -> manipulable manual\nUpload a photo, type a task, **Generate**, then flip / tap / **connect** / add context.")
    steps_state = gr.State([])
    idx_state = gr.State(0)
    base_state = gr.State(None)
    canvas_state = gr.State(None)
    pending_state = gr.State(None)

    with gr.Row():
        with gr.Column(scale=2):
            in_img = gr.Image(type="pil", label="Scene photo")
            task = gr.Textbox(label="Task", placeholder="e.g. Put the laptop on charge, I am in Canada")
            go = gr.Button("Generate manual", variant="primary")
            step_label = gr.Markdown("")
            mode = gr.Radio(["Ask about object", "Connect (press to join)", "Set highlight (correct)"],
                            value="Ask about object", label="Click mode")
            step_view = gr.Image(type="pil", label="Working canvas (click / press here)", interactive=False)
            with gr.Row():
                prev = gr.Button("< Prev")
                nxt = gr.Button("Next >")
                rst = gr.Button("Reset canvas")
        with gr.Column(scale=1):
            gr.Markdown("### Tap-to-deepen")
            deep_img = gr.Image(type="pil", label="Zoom", interactive=False)
            deep_txt = gr.Markdown("")
            gr.Markdown("### Context tray\nUpload a close-up (e.g. a label) for more context.")
            ctx_img = gr.Image(type="pil", label="Extra context image")
            ctx_btn = gr.Button("Add context")
            ctx_note = gr.Markdown("")

    go.click(on_generate, [in_img, task],
             [step_view, step_label, steps_state, idx_state, base_state, canvas_state])
    prev.click(lambda s, i, c: nav(-1, s, i, c), [steps_state, idx_state, canvas_state],
               [step_view, step_label, idx_state])
    nxt.click(lambda s, i, c: nav(1, s, i, c), [steps_state, idx_state, canvas_state],
              [step_view, step_label, idx_state])
    rst.click(reset_canvas, [base_state], [step_view, canvas_state, pending_state])
    step_view.select(on_click, [canvas_state, mode, pending_state],
                     [step_view, deep_img, deep_txt, canvas_state, pending_state])
    ctx_btn.click(on_context, [ctx_img], [ctx_note])

demo.launch(share=True, debug=False)

## How to use

1. Run cells top to bottom (GPU + Internet on).
2. Open the public `gradio.live` link from the last cell.
3. Upload a photo, type a task, click **Generate manual**.
4. **Flip** with Prev/Next to walk the grounded steps.
5. **Connect**: set mode to "Connect (press to join)", press the part to move, then press where it
   joins. The patch snaps onto the destination, the old spot is blurred away, and it is marked joined.
6. **Tap-to-deepen**: mode "Ask about object", click anything to crop + explain it.
7. **Correct**: mode "Set highlight", click where the highlight should be.
8. **Context**: upload a close-up, click **Add context**, read the note.
9. **Reset canvas** restores the original photo.

## What this is and is not
- The "join" is **direct pixel manipulation**, not a generated/edited render. It is a faithful, tiny
  way to feel the interaction (press -> things connect) with no diffusion model.
- To make the join *photorealistic* (the parts truly merged, lighting corrected), you would swap
  `connect()` for an image-editing model. That is your separate, harder faithfulness problem - this
  demo deliberately isolates the **interaction** from the **rendering**.
- Small-model caveat: Moondream's decomposition/detection will sometimes miss. That is expected and
  is exactly why manual connect, correction, and context-upload are in the loop.